In [0]:
# ============================================================
# CELL 1 — IMPORTS + CONFIG
# ============================================================

import os
import datetime as dt
import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql import Window

import mlflow
import mlflow.spark as mlflow_spark
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

os.environ["MLFLOW_DFS_TMP"] = "/Volumes/btc_usd/silver/mlflow_vol"

CATALOG      = "btc_usd"
SILVER_TABLE = f"{CATALOG}.silver.cleaned"
GOLD_TABLE   = f"{CATALOG}.gold.ml_analytics"
EXPERIMENT   = f"/Shared/{CATALOG}_ml"

FEATURE_COLS = [
    "return_1h", "lag_return_2h", "lag_return_3h",
    "lag_return_6h", "lag_return_24h",
    "rsi14", "ma7", "ma21",
    "bb_width", "bb_upper", "bb_lower",
    "volatility_24h", "momentum_6h",
    "volume_ratio", "hour_of_day", "day_of_week", "volume"
]

In [0]:
# ============================================================
# CELL 2 — LOAD DATA
# ============================================================

# Load tables
gold   = spark.table(GOLD_TABLE).orderBy("datetime")
silver = spark.table(SILVER_TABLE).orderBy("datetime")

# Join silver + gold
full = silver.join(
    gold.select("datetime", "prediction", "label",
                "pred_direction", "actual_direction", "correct", "run_id"),
    on="datetime",
    how="inner"
).orderBy("datetime")

# Convert to Pandas for Plotly
gold_pd   = gold.toPandas()
silver_pd = silver.toPandas()
full_pd   = full.toPandas()

# Last 168 hours (1 week)
last_week = full_pd.tail(168)
# Last 72 hours
last_72h  = silver_pd.tail(72)

print(f"✅ Gold   : {len(gold_pd)} rows")
print(f"✅ Silver : {len(silver_pd)} rows")
print(f"✅ Joined : {len(full_pd)} rows")

In [0]:
# ============================================================
# CELL 3 — CHART 1: Model Metrics Summary
# ============================================================

client     = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT)

runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["start_time DESC"],
    max_results=3
)

models      = []
rmse_vals   = []
r2_vals     = []
mae_vals    = []
dir_acc     = []

for run in runs:
    models.append(run.data.params.get("model", "unknown"))
    rmse_vals.append(float(run.data.metrics.get("rmse",               0)))
    r2_vals.append(  float(run.data.metrics.get("r2",                 0)))
    mae_vals.append( float(run.data.metrics.get("mae",                0)))
    dir_acc.append(  float(run.data.metrics.get("direction_accuracy",  0)))

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("RMSE & MAE by Model", "Direction Accuracy by Model")
)

fig.add_trace(go.Bar(name="RMSE", x=models, y=rmse_vals, marker_color="#ef553b"), row=1, col=1)
fig.add_trace(go.Bar(name="MAE",  x=models, y=mae_vals,  marker_color="#636efa"), row=1, col=1)
fig.add_trace(go.Bar(name="Direction Accuracy", x=models, y=dir_acc,
                     marker_color="#00cc96",
                     text=[f"{v:.2%}" for v in dir_acc],
                     textposition="outside"),
              row=1, col=2)

# 50% baseline line
fig.add_hline(y=0.5, line_dash="dash", line_color="red",
              annotation_text="50% baseline", row=1, col=2)

fig.update_layout(
    title="📊 Model Metrics Comparison",
    height=450,
    showlegend=True,
    template="plotly_dark"
)
fig.show()

In [0]:
# ============================================================
# CELL 4 — CHART 2: Actual vs Predicted Returns
# ============================================================

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=last_week["datetime"],
    y=last_week["label"],
    name="Actual Return",
    line=dict(color="#636efa", width=1.5),
    opacity=0.8
))

fig.add_trace(go.Scatter(
    x=last_week["datetime"],
    y=last_week["prediction"],
    name="Predicted Return",
    line=dict(color="#ef553b", width=1.5, dash="dot"),
    opacity=0.8
))

fig.add_hline(y=0, line_dash="dash", line_color="white", opacity=0.3)

fig.update_layout(
    title="📈 Actual vs Predicted Returns (Last 168 Hours)",
    xaxis_title="Datetime",
    yaxis_title="Hourly Return",
    height=450,
    template="plotly_dark",
    hovermode="x unified"
)
fig.show()

In [0]:
# ============================================================
# CELL 5 — CHART 3: Direction Accuracy Over Time
# ============================================================

daily_acc = (
    full.withColumn("date", F.to_date("datetime"))
    .groupBy("date")
    .agg(
        F.round(F.avg("correct"), 4).alias("daily_accuracy"),
        F.count("*").alias("predictions")
    )
    .orderBy("date")
    .toPandas()
)

fig = go.Figure()

# Accuracy line
fig.add_trace(go.Scatter(
    x=daily_acc["date"],
    y=daily_acc["daily_accuracy"],
    name="Daily Direction Accuracy",
    line=dict(color="#00cc96", width=2),
    fill="tozeroy",
    fillcolor="rgba(0, 204, 150, 0.1)"
))

# 50% baseline
fig.add_hline(
    y=0.5,
    line_dash="dash",
    line_color="red",
    annotation_text="50% baseline (coin flip)",
    annotation_position="top right"
)

fig.update_layout(
    title="🎯 Direction Accuracy Over Time (Daily)",
    xaxis_title="Date",
    yaxis_title="Accuracy",
    yaxis=dict(tickformat=".0%", range=[0, 1]),
    height=450,
    template="plotly_dark",
    hovermode="x unified"
)
fig.show()

In [0]:
# ============================================================
# CELL 6 — CHART 4: Candlestick Chart with Volume
# ============================================================

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    row_heights=[0.75, 0.25],
    subplot_titles=("BTC-USD Price (Last 72 Hours)", "Volume")
)

# Candlestick
fig.add_trace(go.Candlestick(
    x=last_72h["datetime"],
    open=last_72h["open"],
    high=last_72h["high"],
    low=last_72h["low"],
    close=last_72h["close"],
    name="OHLC",
    increasing_line_color="#00cc96",
    decreasing_line_color="#ef553b"
), row=1, col=1)

# Bollinger Bands
fig.add_trace(go.Scatter(
    x=last_72h["datetime"], y=last_72h["bb_upper"],
    name="BB Upper", line=dict(color="rgba(255,255,0,0.4)", width=1, dash="dot")
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=last_72h["datetime"], y=last_72h["bb_lower"],
    name="BB Lower", line=dict(color="rgba(255,255,0,0.4)", width=1, dash="dot"),
    fill="tonexty", fillcolor="rgba(255,255,0,0.03)"
), row=1, col=1)

# MA7 and MA21
fig.add_trace(go.Scatter(
    x=last_72h["datetime"], y=last_72h["ma7"],
    name="MA7",  line=dict(color="#636efa", width=1.5)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=last_72h["datetime"], y=last_72h["ma21"],
    name="MA21", line=dict(color="#ab63fa", width=1.5)
), row=1, col=1)

# Volume bars
colors = ["#00cc96" if c >= o else "#ef553b"
          for c, o in zip(last_72h["close"], last_72h["open"])]

fig.add_trace(go.Bar(
    x=last_72h["datetime"],
    y=last_72h["volume"],
    name="Volume",
    marker_color=colors,
    opacity=0.7
), row=2, col=1)

fig.update_layout(
    title="🕯️ BTC-USD Candlestick Chart (Last 72 Hours)",
    height=600,
    template="plotly_dark",
    xaxis_rangeslider_visible=False,
    hovermode="x unified"
)
fig.show()

In [0]:
# ============================================================
# CELL 7 — CHART 5: BTC Price with Buy/Sell Signals
# ============================================================

buy_signals  = last_week[last_week["pred_direction"] == 1]
sell_signals = last_week[last_week["pred_direction"] == 0]
correct      = last_week[last_week["correct"] == 1]
wrong        = last_week[last_week["correct"] == 0]

fig = go.Figure()

# BTC price line
fig.add_trace(go.Scatter(
    x=last_week["datetime"],
    y=last_week["close"],
    name="BTC Price",
    line=dict(color="white", width=1.5),
    opacity=0.8
))

# Buy signals — green triangles up
fig.add_trace(go.Scatter(
    x=buy_signals["datetime"],
    y=buy_signals["close"],
    name="Buy Signal",
    mode="markers",
    marker=dict(symbol="triangle-up", size=10, color="#00cc96"),
))

# Sell signals — red triangles down
fig.add_trace(go.Scatter(
    x=sell_signals["datetime"],
    y=sell_signals["close"],
    name="Sell Signal",
    mode="markers",
    marker=dict(symbol="triangle-down", size=10, color="#ef553b"),
))

# Correct predictions — gold stars
fig.add_trace(go.Scatter(
    x=correct["datetime"],
    y=correct["close"],
    name="Correct ✅",
    mode="markers",
    marker=dict(symbol="star", size=8, color="gold", opacity=0.5),
))

fig.update_layout(
    title="🚦 BTC Price with Buy/Sell Signals (Last 168 Hours)",
    xaxis_title="Datetime",
    yaxis_title="BTC Price (USD)",
    height=500,
    template="plotly_dark",
    hovermode="x unified"
)
fig.show()

In [0]:
# ============================================================
# CELL 8 — CHART 6: Feature Importance
# ============================================================

best_run = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.direction_accuracy DESC"],
    max_results=1
)[0]

best_run_id = best_run.info.run_id
best_model  = best_run.data.params.get("model", "unknown")
print(f"Best model: {best_model} | Run ID: {best_run_id}")

loaded_model = mlflow_spark.load_model(f"runs:/{best_run_id}/model")
regressor    = loaded_model.stages[-1]

if hasattr(regressor, "featureImportances"):
    importances = regressor.featureImportances.toArray()
    fi_df = pd.DataFrame({
        "feature":    FEATURE_COLS,
        "importance": importances
    }).sort_values("importance", ascending=True)

    fig = go.Figure(go.Bar(
        x=fi_df["importance"],
        y=fi_df["feature"],
        orientation="h",
        marker_color="#636efa"
    ))

elif hasattr(regressor, "coefficients"):
    import numpy as np
    coefficients = regressor.coefficients.toArray()
    fi_df = pd.DataFrame({
        "feature":     FEATURE_COLS,
        "importance":  abs(coefficients)
    }).sort_values("importance", ascending=True)

    fig = go.Figure(go.Bar(
        x=fi_df["importance"],
        y=fi_df["feature"],
        orientation="h",
        marker_color="#636efa"
    ))

fig.update_layout(
    title=f"🔍 Feature Importance — {best_model.upper()}",
    xaxis_title="Importance",
    yaxis_title="Feature",
    height=550,
    template="plotly_dark",
    margin=dict(l=150)
)
fig.show()

In [0]:
# ============================================================
# CELL 9 — SECTION 7: Overall Summary Stats
# ============================================================

print("=" * 55)
print("📋 OVERALL SUMMARY")
print("=" * 55)

gold.select(
    F.count("*").alias("total_predictions"),
    F.round(F.avg("correct"),              4).alias("overall_direction_accuracy"),
    F.round(F.sum("correct"),              0).alias("correct_predictions"),
    F.round(F.avg("prediction"),           6).alias("avg_predicted_return"),
    F.round(F.avg("label"),                6).alias("avg_actual_return"),
    F.round(F.max("close"),                2).alias("btc_high"),
    F.round(F.min("close"),                2).alias("btc_low"),
    F.min("datetime").alias("from"),
    F.max("datetime").alias("to")
).display()